# Curadoria do dataset Brasa — v001

Este notebook audita o snapshot do Drive, cria um inventário de fontes para revisão humana e só exporta exemplos quando a fonte foi aprovada explicitamente. Ele não instala pacotes, consulta APIs externas nem treina modelo.

**Proteções:** `corpus/eval/` fica reservado para avaliação; `corpus/quarantine/` nunca é elegível. Nenhuma fonte começa aprovada. Conhecimento bruto, índices e arquivos de pesquisa não viram automaticamente exemplos de programação. Aprovar uma fonte exige revisar licença/proveniência e qualidade.

Execute as células em ordem. A fonte de verdade permanece no Drive; o espaço temporário usado pelo runtime do Colab pode ser apagado quando a sessão termina.


In [ ]:
from google.colab import drive
from pathlib import Path
import csv
import hashlib
import json
import re
import shutil
from collections import Counter
from datetime import datetime, timezone

# Monte o Drive e localize o snapshot imutável enviado anteriormente.
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/IA-Local/datasets')
SNAPSHOT_VERSION = 'v001'
SNAPSHOT = DRIVE / 'raw' / SNAPSHOT_VERSION / 'snapshot'
CURATION = DRIVE / 'curation' / SNAPSHOT_VERSION
CURATION.mkdir(parents=True, exist_ok=True)

assert SNAPSHOT.is_dir(), f'Snapshot não encontrado: {SNAPSHOT}'
MANIFEST_PATH = SNAPSHOT / 'MANIFEST.json'
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
assert manifest.get('schema') == 'ia-local-dataset-snapshot/v1'
assert manifest.get('version') == SNAPSHOT_VERSION
print('Snapshot:', SNAPSHOT)
print('Arquivos no manifesto:', manifest['file_count'])
print('Quarentena permanece fora de treino:', 'quarantine' in manifest.get('training_policy', '').lower())


In [ ]:
# Confere integridade e correspondência entre manifesto e arquivos extraídos.
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

manifest_entries = {item['path']: item for item in manifest['files']}
actual_paths = set()
for root_name in ('corpus', 'python/data'):
    root = SNAPSHOT / root_name
    assert root.is_dir(), f'Diretório ausente: {root}'
    for path in root.rglob('*'):
        if path.is_file():
            relative = path.relative_to(SNAPSHOT).as_posix()
            actual_paths.add(relative)

missing = sorted(set(manifest_entries) - actual_paths)
unlisted = sorted(actual_paths - set(manifest_entries))
assert not missing, f'Arquivos do manifesto ausentes: {missing[:5]}'
assert not unlisted, f'Arquivos sem entrada no manifesto: {unlisted[:5]}'

integrity_rows = []
for relative, expected in manifest_entries.items():
    path = SNAPSHOT / relative
    actual_size = path.stat().st_size
    actual_hash = sha256_file(path)
    assert actual_size == expected['bytes'], f'Tamanho divergente: {relative}'
    assert actual_hash == expected['sha256'], f'Hash divergente: {relative}'
    integrity_rows.append({'path': relative, 'bytes': actual_size, 'sha256': actual_hash})

print(f'Integridade conferida: {len(integrity_rows)} arquivos; {sum(r["bytes"] for r in integrity_rows):,} bytes.')


## Auditoria de formatos e fontes

A próxima célula conta registros, erros de leitura, campos mais comuns e sinais de licença/proveniência. Ela registra hashes e estrutura, sem copiar textos das conversas para o relatório. A aprovação é por arquivo e começa pendente; o exportador aceita somente arquivos JSONL explicitamente liberados.


In [ ]:
def classify_source(relative):
    parts = relative.split('/')
    name = parts[-1].lower()
    if 'quarantine' in parts:
        return 'quarantine'
    if 'eval' in parts or 'heldout' in name or 'holdout' in name:
        return 'evaluation_holdout'
    if relative.startswith('corpus/training/') and name.endswith('.jsonl'):
        return 'training_candidate'
    if relative.startswith('python/data/') and name.endswith('.jsonl'):
        return 'training_candidate'
    if relative.startswith('corpus/clean/'):
        return 'knowledge_candidate'
    if relative.startswith('corpus/raw/'):
        return 'raw_source'
    return 'metadata_or_index'

LICENSE_FIELDS = {'license', 'license_policy', 'source', 'source_id', 'source_repo', 'provenance'}
def iter_json_records(path, relative):
    if path.suffix.lower() == '.jsonl':
        with path.open('rb') as stream:
            for line_number, raw in enumerate(stream, 1):
                if not raw.strip():
                    continue
                try:
                    yield line_number, json.loads(raw.decode('utf-8')), None
                except Exception as error:
                    yield line_number, None, type(error).__name__
    elif path.suffix.lower() == '.json' and path.stat().st_size <= 32 * 1024 * 1024:
        try:
            value = json.loads(path.read_text(encoding='utf-8'))
            rows = value if isinstance(value, list) else [value]
            for index, row in enumerate(rows, 1):
                yield index, row, None
        except Exception as error:
            yield 1, None, type(error).__name__

inventory = []
heldout_prompt_hashes = set()
training_row_hashes = set()
for item in integrity_rows:
    relative = item['path']
    path = SNAPSHOT / relative
    category = classify_source(relative)
    suffix = path.suffix.lower()
    row_count = 0
    invalid_rows = 0
    skipped_large_json = suffix == '.json' and path.stat().st_size > 32 * 1024 * 1024
    key_counts = Counter()
    license_signals = Counter()
    duplicate_rows = 0
    local_hashes = set() if category == 'training_candidate' else None

    if suffix in {'.jsonl', '.json'} and not skipped_large_json:
        for line_number, row, error_name in iter_json_records(path, relative):
            if error_name:
                invalid_rows += 1
                continue
            row_count += 1
            if isinstance(row, dict):
                key_counts.update(row.keys())
                for field in LICENSE_FIELDS:
                    value = row.get(field)
                    if isinstance(value, (str, int, float)) and str(value).strip():
                        license_signals[f'{field}={str(value)[:100]}'] += 1
                if category == 'training_candidate':
                    fingerprint = hashlib.sha256(json.dumps(row, ensure_ascii=False, sort_keys=True, separators=(',', ':')).encode('utf-8')).hexdigest()
                    if fingerprint in local_hashes:
                        duplicate_rows += 1
                    local_hashes.add(fingerprint)
                if category == 'evaluation_holdout':
                    for field in ('prompt', 'question', 'request', 'instruction'):
                        text = row.get(field)
                        if isinstance(text, str) and text.strip():
                            normalized = ' '.join(text.casefold().split())
                            heldout_prompt_hashes.add(hashlib.sha256(normalized.encode('utf-8')).hexdigest())

    inventory.append({
        **item,
        'category': category,
        'format': suffix or path.suffix,
        'records': row_count,
        'invalid_records': invalid_rows,
        'duplicate_records_within_file': duplicate_rows,
        'skipped_large_json': skipped_large_json,
        'top_keys': dict(key_counts.most_common(12)),
        'license_signals': dict(license_signals.most_common(12)),
    })

category_counts = Counter(row['category'] for row in inventory)
print('Arquivos por categoria:', dict(category_counts))
print('Prompts de avaliação reservados (hashes exatos):', len(heldout_prompt_hashes))
print('Arquivos com JSON/JSONL inválido:', sum(row['invalid_records'] > 0 for row in inventory))
print('Linhas duplicadas em arquivos candidatos:', sum(row['duplicate_records_within_file'] for row in inventory))


In [ ]:
# Persiste inventário e relatório de auditoria no Drive.
# O registro de aprovação existente é preservado quando o hash da fonte não mudou.
registry_path = CURATION / 'source_approval_registry.csv'
old_rows = {}
if registry_path.exists():
    with registry_path.open(encoding='utf-8', newline='') as stream:
        for row in csv.DictReader(stream):
            old_rows[(row.get('source_path'), row.get('sha256'))] = row

registry_fields = [
    'source_path', 'sha256', 'records', 'top_keys', 'license_signals',
    'training_decision', 'license_decision', 'quality_decision', 'reviewer_notes'
]
registry_rows = []
for row in inventory:
    if row['category'] != 'training_candidate':
        continue
    previous = old_rows.get((row['path'], row['sha256']), {})
    registry_rows.append({
        'source_path': row['path'],
        'sha256': row['sha256'],
        'records': row['records'],
        'top_keys': json.dumps(row['top_keys'], ensure_ascii=False, sort_keys=True),
        'license_signals': json.dumps(row['license_signals'], ensure_ascii=False, sort_keys=True),
        'training_decision': previous.get('training_decision', 'pending'),
        'license_decision': previous.get('license_decision', 'pending'),
        'quality_decision': previous.get('quality_decision', 'pending'),
        'reviewer_notes': previous.get('reviewer_notes', ''),
    })

with registry_path.open('w', encoding='utf-8', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=registry_fields)
    writer.writeheader()
    writer.writerows(registry_rows)

inventory_path = CURATION / 'file_inventory.csv'
with inventory_path.open('w', encoding='utf-8', newline='') as stream:
    fields = ['path', 'category', 'format', 'bytes', 'sha256', 'records', 'invalid_records', 'duplicate_records_within_file', 'skipped_large_json', 'top_keys', 'license_signals']
    writer = csv.DictWriter(stream, fieldnames=fields)
    writer.writeheader()
    for row in inventory:
        writer.writerow({**row, 'top_keys': json.dumps(row['top_keys'], ensure_ascii=False, sort_keys=True), 'license_signals': json.dumps(row['license_signals'], ensure_ascii=False, sort_keys=True)})

summary = {
    'schema': 'brasa-dataset-audit/v1',
    'snapshot_version': SNAPSHOT_VERSION,
    'snapshot_manifest_sha256': sha256_file(MANIFEST_PATH),
    'created_at': datetime.now(timezone.utc).isoformat(),
    'file_count': len(inventory),
    'record_count_parsed': sum(row['records'] for row in inventory),
    'invalid_record_count': sum(row['invalid_records'] for row in inventory),
    'category_counts': dict(category_counts),
    'heldout_prompt_hash_count': len(heldout_prompt_hashes),
    'training_candidates_pending_review': len(registry_rows),
    'policy': 'Evaluation and quarantine are excluded; no source is approved automatically.',
}
(CURATION / 'audit_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('Relatórios salvos em:', CURATION)
print('Registro de aprovação:', registry_path)
print('Fontes candidatas:', len(registry_rows))


## Revisão por registro e primeiro lote

O registro de fontes continua exigindo procedência e licença. A célula seguinte cria também `record_review_registry.csv`: cada exemplo começa pendente. Foram revisados 40 exemplos de dois arquivos que o manifesto do próprio projeto identifica como autoria local determinística. A seleção inclui práticas de programação, depuração, uso de ferramentas e verificação. O exemplo que exigia aprovação antes de **qualquer** escrita fica rejeitado.

Cinco arquivos ficam marcados como rejeitados para esta exportação: um arquivo vazio, cópias repetidas e sequências com resultados de ferramenta simulados. As demais fontes seguem pendentes. Todas as decisões usam caminho, hash do arquivo, número da linha e hash do registro; uma alteração de conteúdo invalida a seleção.

Execute esta célula antes da exportação. Revise os CSVs no Drive para ampliar ou corrigir o lote. A saída é uma primeira amostra de comportamento do agente; ainda não reúne tarefas de programação suficientes para treinar um programador funcional.


In [ ]:
# Aplica a revisão inicial às fontes locais e exige aprovação individual para cada exemplo.
REVIEWED_COHORT = {
    'python/data/godmode_knowledge_v1.jsonl': {
        'sha256': 'c5beae25fa29037301b17bfeb976b432b7cbe9b4bf17a9c1248c57df696a0736',
        'lines': {8, 9, *range(11, 21), 21, 23, 27, 28, 29, 30, 31, 32, 34, 35, 37, 38, 40, 41, 43, 48, 50},
    },
    'python/data/godmode_procedures_v1.jsonl': {
        'sha256': '9d272d64706227bc39aea63c2b322c0d0fb0356ca91ce820b009328f23a118eb',
        'lines': {1, 2, 5, 8, 11, 16, 17, 22, 27, 29, 30},
    },
}

source_evidence = json.loads((SNAPSHOT / 'python/data/godmode_datasets_manifest_v1.json').read_text(encoding='utf-8'))
assert 'autoria local determinística' in source_evidence.get('provenance', '').casefold()
for info in REVIEWED_COHORT.values():
    assert any(part.get('sha256') == info['sha256'] for part in source_evidence['training'].values())
for source_path, info in REVIEWED_COHORT.items():
    assert manifest_entries[source_path]['sha256'] == info['sha256']

REJECTED_SOURCES = {
    'corpus/training/teacher_plans_qwen3_30b.jsonl': 'arquivo vazio',
    'python/data/behavior_augmented.jsonl': 'resultados de ferramenta simulados apresentados como evidência',
    'python/data/behavior_augmented_v2.jsonl': 'cópia idêntica de behavior_augmented.jsonl',
    'python/data/combined.jsonl': 'repete exemplos de behavior_augmented e tool_traces',
    'python/data/tool_traces.jsonl': 'resultados de ferramenta simulados e exemplos repetidos',
}

def append_note(previous, note):
    previous = (previous or '').strip()
    return previous if note in previous else (previous + ' | ' + note).strip(' |')

with registry_path.open(encoding='utf-8', newline='') as stream:
    source_reviews = list(csv.DictReader(stream))
for review in source_reviews:
    source_path = review['source_path']
    if source_path in REVIEWED_COHORT and review['sha256'] == REVIEWED_COHORT[source_path]['sha256']:
        for field, value in (('training_decision', 'approve'), ('license_decision', 'approved'), ('quality_decision', 'approved')):
            if review.get(field, '').strip().lower() in {'', 'pending'}:
                review[field] = value
        review['reviewer_notes'] = append_note(review.get('reviewer_notes'), 'Manifesto local de autoria verificado por hash; somente registros selecionados passam pela revisão individual.')
    elif source_path in REJECTED_SOURCES:
        if review.get('training_decision', '').strip().lower() in {'', 'pending'}:
            review['training_decision'] = 'reject'
        if review.get('quality_decision', '').strip().lower() in {'', 'pending'}:
            review['quality_decision'] = 'rejected'
        review['reviewer_notes'] = append_note(review.get('reviewer_notes'), REJECTED_SOURCES[source_path])

with registry_path.open('w', encoding='utf-8', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=registry_fields)
    writer.writeheader()
    writer.writerows(source_reviews)

record_registry_path = CURATION / 'record_review_registry.csv'
old_record_reviews = {}
if record_registry_path.exists():
    with record_registry_path.open(encoding='utf-8', newline='') as stream:
        for row in csv.DictReader(stream):
            key = (row.get('source_path'), row.get('source_file_sha256'), row.get('line_number'), row.get('record_sha256'))
            old_record_reviews[key] = row

record_fields = [
    'source_path', 'source_file_sha256', 'line_number', 'record_sha256',
    'domain', 'kind', 'prompt_preview', 'answer_preview', 'review_decision', 'review_notes'
]
record_reviews = []
found_selected = set()
for file_info in inventory:
    if file_info['category'] != 'training_candidate':
        continue
    source_path = file_info['path']
    for line_number, row, error_name in iter_json_records(SNAPSHOT / source_path, source_path):
        if error_name or not isinstance(row, dict):
            continue
        record_hash = hashlib.sha256(json.dumps(row, ensure_ascii=False, sort_keys=True, separators=(',', ':')).encode('utf-8')).hexdigest()
        key = (source_path, file_info['sha256'], str(line_number), record_hash)
        previous = old_record_reviews.get(key, {})
        messages = row.get('messages', [])
        if isinstance(messages, list):
            prompt = ' '.join(str(m.get('content', '')) for m in messages if isinstance(m, dict) and m.get('role') == 'user')
            answer = ' '.join(str(m.get('content', '')) for m in messages if isinstance(m, dict) and m.get('role') == 'assistant')
        else:
            prompt = str(row.get('instruction', ''))
            answer = str(row.get('output', ''))
        decision = previous.get('review_decision', 'pending')
        note = previous.get('review_notes', '')
        if source_path in REVIEWED_COHORT and line_number in REVIEWED_COHORT[source_path]['lines']:
            found_selected.add((source_path, line_number))
            if decision.strip().lower() in {'', 'pending'}:
                decision = 'approve'
            note = append_note(note, 'Revisão inicial: comportamento útil, autoria local documentada e sem exigência geral de aprovação para escrita.')
        elif source_path == 'python/data/godmode_knowledge_v1.jsonl' and line_number == 39:
            if decision.strip().lower() in {'', 'pending'}:
                decision = 'reject'
            note = append_note(note, 'Regra incompatível: exige aprovação antes de qualquer escrita em arquivo.')
        record_reviews.append({
            'source_path': source_path,
            'source_file_sha256': file_info['sha256'],
            'line_number': line_number,
            'record_sha256': record_hash,
            'domain': str(row.get('domain', '')),
            'kind': str(row.get('kind', '')),
            'prompt_preview': ' '.join(prompt.split())[:160],
            'answer_preview': ' '.join(answer.split())[:180],
            'review_decision': decision,
            'review_notes': note,
        })

expected_selected = {(path, line) for path, info in REVIEWED_COHORT.items() for line in info['lines']}
assert found_selected == expected_selected, 'O lote revisado não corresponde ao snapshot esperado.'
with record_registry_path.open('w', encoding='utf-8', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=record_fields)
    writer.writeheader()
    writer.writerows(record_reviews)
summary_path = CURATION / 'audit_summary.json'
updated_summary = json.loads(summary_path.read_text(encoding='utf-8'))
updated_summary.pop('training_candidates_pending_review', None)
updated_summary['source_review_counts'] = dict(Counter(row['training_decision'] for row in source_reviews))
updated_summary['record_review_counts'] = dict(Counter(row['review_decision'] for row in record_reviews))
summary_path.write_text(json.dumps(updated_summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('Fontes com revisão inicial:', len(REVIEWED_COHORT))
print('Registros individuais revisados:', len(expected_selected))
print('Registro individual:', record_registry_path)


In [ ]:
# Releia as aprovações editadas no Drive e exporte um run imutável, se houver fontes liberadas.
with registry_path.open(encoding='utf-8', newline='') as stream:
    approved_sources = [row for row in csv.DictReader(stream)
        if row.get('training_decision', '').strip().lower() == 'approve'
        and row.get('license_decision', '').strip().lower() == 'approved'
        and row.get('quality_decision', '').strip().lower() == 'approved']

with record_registry_path.open(encoding='utf-8', newline='') as stream:
    approved_record_keys = {
        (row['source_path'], row['source_file_sha256'], int(row['line_number']), row['record_sha256'])
        for row in csv.DictReader(stream)
        if row.get('review_decision', '').strip().lower() == 'approve'
    }

if not approved_sources or not approved_record_keys:
    print('Nenhum registro tem aprovação da fonte e revisão individual. Nenhum dataset foi exportado.')
else:
    def normalize_record(row):
        if isinstance(row.get('messages'), list):
            messages = row['messages']
        elif isinstance(row.get('instruction'), str) and isinstance(row.get('output'), str):
            messages = []
            if isinstance(row.get('system'), str) and row['system'].strip():
                messages.append({'role': 'system', 'content': row['system']})
            messages.extend([
                {'role': 'user', 'content': row['instruction']},
                {'role': 'assistant', 'content': row['output']},
            ])
        else:
            return None, 'unsupported_schema'

        allowed_roles = {'system', 'user', 'assistant', 'tool'}
        if not messages or any(not isinstance(message, dict) or message.get('role') not in allowed_roles for message in messages):
            return None, 'invalid_messages'
        clean_messages = []
        for message in messages:
            clean = {key: message[key] for key in ('role', 'content', 'tool_call', 'tool_calls', 'name', 'citations') if key in message}
            if not any(key in clean for key in ('content', 'tool_call', 'tool_calls')):
                return None, 'empty_message'
            clean_messages.append(clean)
        user_text = '\n'.join(str(message.get('content', '')) for message in clean_messages if message['role'] == 'user').strip()
        has_assistant = any(message['role'] == 'assistant' and (str(message.get('content', '')).strip() or 'tool_call' in message or 'tool_calls' in message) for message in clean_messages)
        if not user_text or not has_assistant:
            return None, 'missing_user_or_assistant'
        return clean_messages, None

    approved_by_path = {row['source_path']: row for row in approved_sources}
    candidate_by_path = {row['path']: row for row in inventory if row['category'] == 'training_candidate'}
    train_rows, validation_rows = [], []
    seen_examples = set()
    excluded = Counter()
    source_hashes = {}
    prompt_groups = {}

    for source_path in sorted(approved_by_path):
        approval = approved_by_path[source_path]
        inventory_row = candidate_by_path.get(source_path)
        if inventory_row is None or inventory_row['sha256'] != approval['sha256']:
            excluded['source_missing_or_changed_since_audit'] += 1
            continue
        path = SNAPSHOT / source_path
        if sha256_file(path) != approval['sha256']:
            excluded['source_hash_mismatch'] += 1
            continue
        source_hashes[source_path] = approval['sha256']
        for line_number, row, error_name in iter_json_records(path, source_path):
            if error_name:
                excluded['invalid_json'] += 1
                continue
            if not isinstance(row, dict):
                excluded['not_an_object'] += 1
                continue
            source_record_hash = hashlib.sha256(json.dumps(row, ensure_ascii=False, sort_keys=True, separators=(',', ':')).encode('utf-8')).hexdigest()
            if (source_path, approval['sha256'], line_number, source_record_hash) not in approved_record_keys:
                excluded['record_not_approved'] += 1
                continue
            marker = str(row.get('kind', '')).strip().lower()
            split_marker = str(row.get('split', row.get('partition', ''))).strip().lower()
            if marker in {'heldout', 'holdout', 'evaluation', 'eval'} or split_marker in {'heldout', 'holdout', 'evaluation', 'eval', 'test'}:
                excluded['record_marked_evaluation'] += 1
                continue
            messages, reason = normalize_record(row)
            if reason:
                excluded[reason] += 1
                continue
            prompt_text = ' '.join(' '.join(str(message.get('content', '')).casefold().split()) for message in messages if message['role'] == 'user').strip()
            prompt_hash = hashlib.sha256(prompt_text.encode('utf-8')).hexdigest()
            if prompt_hash in heldout_prompt_hashes:
                excluded['exact_heldout_prompt_match'] += 1
                continue
            serialized = json.dumps(messages, ensure_ascii=False, sort_keys=True, separators=(',', ':'))
            example_hash = hashlib.sha256(serialized.encode('utf-8')).hexdigest()
            if example_hash in seen_examples:
                excluded['exact_duplicate'] += 1
                continue
            seen_examples.add(example_hash)
            example = {
                'schema': 'brasa-sft-example/v1',
                'messages': messages,
                'metadata': {
                    'source_path': source_path,
                    'source_line': line_number,
                    'source_file_sha256': approval['sha256'],
                    'source_record_sha256': source_record_hash,
                    'normalized_example_sha256': example_hash,
                    'license_review': 'approved',
                },
            }
            prompt_groups.setdefault(prompt_hash, []).append(example)

    # O mesmo prompt fica sempre no mesmo split, evitando vazamento literal entre treino e validação.
    for prompt_hash, examples in sorted(prompt_groups.items()):
        bucket = int(prompt_hash[:8], 16) % 10
        (validation_rows if bucket == 0 else train_rows).extend(examples)

    if not train_rows and not validation_rows:
        print('As fontes aprovadas não produziram exemplos válidos; nada foi exportado.')
        print('Exclusões por motivo:', dict(excluded))
    else:
        run_id = datetime.now(timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')
        output_dir = DRIVE / 'curated' / SNAPSHOT_VERSION / 'runs' / run_id
        output_dir.mkdir(parents=True, exist_ok=False)
        def write_jsonl(path, rows):
            with path.open('w', encoding='utf-8') as stream:
                for row in rows:
                    stream.write(json.dumps(row, ensure_ascii=False) + '\n')
        write_jsonl(output_dir / 'train.jsonl', train_rows)
        write_jsonl(output_dir / 'validation.jsonl', validation_rows)
        output_manifest = {
            'schema': 'brasa-curated-dataset/v1',
            'snapshot_version': SNAPSHOT_VERSION,
            'created_at': datetime.now(timezone.utc).isoformat(),
            'source_manifest_sha256': sha256_file(MANIFEST_PATH),
            'source_files': source_hashes,
            'approval_registry_sha256': sha256_file(registry_path),
            'record_review_registry_sha256': sha256_file(record_registry_path),
            'train_records': len(train_rows),
            'validation_records': len(validation_rows),
            'split': 'stable 90/10 by sha256 of normalized user prompt; exact prompt groups stay together',
            'excluded_counts': dict(excluded),
            'evaluation_policy': 'corpus/eval is never eligible; exact normalized prompt matches are excluded',
        }
        (output_dir / 'MANIFEST.json').write_text(json.dumps(output_manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
        print('Dataset exportado:', output_dir)
        print('Treino:', len(train_rows), '| Validação:', len(validation_rows))
        print('Exclusões:', dict(excluded))
        if not validation_rows:
            print('Aviso: a divisão não produziu validação; aprove mais exemplos com prompts distintos antes de treinar.')
